In [92]:
import os
import glob
import pandas as pd
import whisper
from jiwer import wer, cer, Compose, ToLowerCase, RemovePunctuation, Strip, RemoveMultipleSpaces
from pathlib import Path

DATA_ROOT = os.path.join("data", "ami")

AUDIO_DIR = os.path.join(DATA_ROOT, "audio")
SUMMARY_DIR = os.path.join(DATA_ROOT, "summary")
TRANSCRIPT_DIR = os.path.join(DATA_ROOT, "transcript")

In [93]:
def parse_transcript(filepath):

    utterances = []

    with open(filepath, "r", encoding="utf-8") as f:

        for line in f:

            parts = line.strip().split("\t")

            if len(parts) != 4:
                print('Transcript line parts mismatch')
                continue

            speaker, start_time, text, section = parts

            utterances.append({
                "speaker": speaker,
                "start_time": float(start_time),
                "text": text,
                "section": section
            })

    utterance_df = pd.DataFrame(utterances)

    if len(utterance_df) == 0:
        return "", utterance_df

    # Build full transcript text
    transcript_text = " ".join(utterance_df["text"].tolist())

    return transcript_text, utterance_df

In [94]:
def load_transcripts(transcript_dir):

    records = []

    for split in ["train", "valid", "test"]:

        split_dir = os.path.join(transcript_dir, split)

        for filepath in glob.glob(os.path.join(split_dir, "*.txt")):

            meeting_id = os.path.basename(filepath).replace(".txt", "")

            transcript_text, utterance_df = parse_transcript(filepath)

            records.append({
                "meeting_id": meeting_id,
                "split": split,
                "transcript": transcript_text,
                "num_utterances": len(utterance_df),
                "num_speakers": utterance_df["speaker"].nunique()
            })

    return pd.DataFrame(records)

In [95]:
def load_summaries(summary_dir):

    summaries = {}

    for filepath in glob.glob(os.path.join(summary_dir, "*.txt")):
        meeting_id = os.path.basename(filepath).replace(".txt", "")

        with open(filepath, "r", encoding="utf-8") as f:
            summaries[meeting_id] = f.read()

    return summaries

In [96]:
def load_audio_paths(audio_dir):

    audio_records = []

    for meeting in os.listdir(audio_dir):

        meeting_audio_dir = os.path.join(audio_dir, meeting, "audio")

        if not os.path.isdir(meeting_audio_dir):
            continue

        headset = None
        lapel = None

        for file in os.listdir(meeting_audio_dir):

            if "Headset" in file:
                headset = os.path.join(meeting_audio_dir, file)

            if "Lapel" in file:
                lapel = os.path.join(meeting_audio_dir, file)

        audio_records.append({
            "meeting_id": meeting,
            "audio_headset": headset,
            "audio_lapel": lapel
        })

    return pd.DataFrame(audio_records)

In [97]:
def build_dataset():
    transcripts_df = load_transcripts(TRANSCRIPT_DIR)

    summaries = load_summaries(SUMMARY_DIR)

    audio_df = load_audio_paths(AUDIO_DIR)

    # attach summaries
    transcripts_df["summary"] = transcripts_df["meeting_id"].map(summaries)

    # merge audio paths
    df = transcripts_df.merge(audio_df, on="meeting_id", how="left")
    
    # get the transcript and summary lengths in tokens, and the ratio of summary length to transcript length
    # we also tested character counts, but the ratios were very similar to token counts
    df['transcript_length'] = df['transcript'].apply(lambda x: len(x.split()) if isinstance(x, str) else 0)
    df['summary_length'] = df['summary'].apply(lambda x: len(x.split()) if isinstance(x, str) else 0)
    df['ratio_summary_to_transcript'] = df.apply(lambda row: row['summary_length'] / row['transcript_length'] if row['transcript_length'] > 0 else 0, axis=1)

    return df

In [98]:
df = build_dataset()

#print(df.shape)
#print(df.head())

#model = whisper.load_model("base")

#audio = df["audio_headset"]
p = r"C:\Users\culle\Downloads\AI-574-Group4-Main"
#for a in audio:
#    file_path = os.path.join(p, a)
#    print(file_path)
#    result = model.transcribe(file_path)
#    print(result["text"])

#    a_path = os.path.splitext(os.path.basename(file_path))[0]
#    output_path = os.path.join(p + "\Text\Tran" + a_path + ".txt")
#    with open(output_path, "w", encoding = "utf-8") as f:
#        f.write(result["text"])

whisper_path = os.path.join(p + "\Text")
transcript_path = r"C:\Users\culle\Downloads\AI-574-Group4-Main\Transcripts"

transform = Compose([ToLowerCase(), RemovePunctuation(), Strip(), RemoveMultipleSpaces()])

fillers = {"uh", "um", "uhh", "umm", "hmm", "ah", "oh", "er", "mm", "mhm"}

def remove_fillers(text):
    words = text.split()
    words = [w for w in words if w not in fillers]

    return " ".join(words)

results = []

for f in os.listdir(whisper_path):
    if f.endswith(".txt"):
        w_path = os.path.join(whisper_path, f)
        t_path = os.path.join(transcript_path, f)

        whis = ""
        trans = ""
        with open(w_path, "r", encoding = "utf-8") as f:
            whis = f.read()

        with open(t_path, "r", encoding = "utf-8") as f:
            tran = f.read()

        normal_whis = transform(whis)
        normal_tran = transform(tran)

        normal_whis = remove_fillers(normal_whis)
        normal_tran = remove_fillers(normal_tran)

        #print(Path(f.name).stem, len(normal_whis.split()), len(normal_tran.split()), len(normal_whis.split()) / len(normal_tran.split()))

        whis_wer = wer(normal_tran, normal_whis)
        whis_cer = cer(normal_tran, normal_whis)

        results.append((Path(f.name).stem, whis_wer, whis_cer))

print(results)

[('TranES2002a.Mix-Headset', 0.8434077079107505, 0.6540905480540111), ('TranES2002b.Mix-Headset', 0.9137931034482759, 0.6890987683562293), ('TranES2002c.Mix-Headset', 0.9289163090128756, 0.6998069498069498), ('TranES2002d.Mix-Headset', 0.9132370800161922, 0.6897451064745106), ('TranES2003a.Mix-Headset', 0.8969603297269448, 0.6862942122186495), ('TranES2003b.Mix-Headset', 0.9297436843075788, 0.7080577918343876), ('TranES2003c.Mix-Headset', 0.8882435762055614, 0.6935101186322401), ('TranES2003d.Mix-Headset', 0.994536452108588, 0.9221597916952172), ('TranES2004a.Mix-Headset', 0.8641239570917759, 0.6679503300848789), ('TranES2004b.Mix-Headset', 0.9216589861751152, 0.697941598851125), ('TranES2004c.Mix-Headset', 0.899911426040744, 0.6955438287892114), ('TranES2004d.Mix-Headset', 0.9207837883101658, 0.6915064102564102), ('TranES2005a.Mix-Headset', 0.7254901960784313, 0.5368047982551799), ('TranES2005b.Mix-Headset', 0.8815360877764443, 0.6560455863510688), ('TranES2005c.Mix-Headset', 0.897190